# Cell embedding with MultiVI

Bolero conditions every prediction on an **atlas-scale cell-state embedding**, and it is
trained on **metacells** rather than noisy single cells. Two tutorial pages build those
inputs for a dataset, using **ChromiumPBMC** as the worked example:

1. **This page** — turn a single-cell multiome dataset into a low-dimensional *cell
   embedding* with a joint RNA + ATAC **MultiVI** model.
2. [Meta cells](02_meta_cell.ipynb) — collapse cells into SEACells *metacells* on top of
   that embedding.

> **Where the data comes from.** Datasets are registered in the companion package
> [`bolerodata`](https://github.com/liuhlab/bolerodata). `DATASETS["ChromiumPBMC"]` resolves
> the on-disk single-cell RNA / ATAC `AnnData` files and the standardized cell metadata from
> the lab data lake, so you never manage file paths by hand.

> **Prerequisites.** A CUDA GPU, the `bolerodata` data lake mounted, and the bolero runtime
> environment (`pixi shell`). Steps 3–5 are heavy (a full MultiVI fit over the whole
> dataset). Point `WORK_DIR` at an existing results directory to **reuse** cached artifacts,
> or at a fresh directory to run the whole pipeline end-to-end.

## Setup

In [ ]:
from pathlib import Path

import anndata
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import seaborn as sns
import torch
from bolerodata import DATASETS
from scipy.sparse import csr_matrix
from scvi.external import POISSONMULTIVI
from tqdm.auto import tqdm

scvi.settings.seed = 0
torch.set_float32_matmul_precision("high")
sc.set_figure_params(figsize=(4, 4), frameon=False)

In [ ]:
# --- Configuration ---------------------------------------------------------------
DATASET_NAME = "ChromiumPBMC"

# All heavy intermediate artifacts are written here (NOT into the docs tree).
# Reuse an existing results directory to skip recomputation, or use a fresh empty
# directory to run the whole pipeline from scratch.
WORK_DIR = Path(
    "/large_storage/zhoulab/hanliu/wmb/standard/embedding/poissonmultivi/ChromiumPBMC"
)
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Cap cells per (subclass x tissue) group so a few abundant states do not dominate
# training of the embedding model.
EXP_BATCH = "donor"                        # experimental-batch covariate to regress out
DOWNSAMPLE_GROUPBY = ["subclass", "tissue"]
MAX_CELLS_PER_GROUP = 1500

N_TOP_GENES = 5000                         # highly-variable genes kept for the RNA view
N_LATENT = 30                              # cell-embedding dimensionality
MAX_EPOCHS = 200                           # lower (e.g. 10) for a quick smoke test

dataset = DATASETS[DATASET_NAME]
dataset.name

## Step 1 — Build the training cell list

Start from the standardized per-cell metadata and:

- drop extreme-coverage outliers (likely multiplets / debris), then
- **downsample** each `subclass x tissue` group to at most `MAX_CELLS_PER_GROUP` cells.

The downsampled list is used only to *train* the embedding model efficiently; the trained
model is later applied to **all** cells (Step 5).

In [ ]:
cell_metadata = dataset.make_cell_metadata()

# Drop extreme-coverage outliers (2x the 99th percentile of UMIs / fragments).
max_umi = cell_metadata["n_umis"].quantile(0.99) * 2
max_frag = cell_metadata["n_fragments"].quantile(0.99) * 2
keep = (cell_metadata["n_umis"] < max_umi) & (cell_metadata["n_fragments"] < max_frag)
cell_metadata = cell_metadata[keep].copy()
print(f"{keep.sum()} / {keep.size} cells pass the coverage filter")

fig, axes = plt.subplots(figsize=(8, 2), nrows=2, constrained_layout=True)
sns.histplot(cell_metadata["n_umis"], bins=100, ax=axes[0])
sns.histplot(cell_metadata["n_fragments"], bins=100, ax=axes[1])

In [ ]:
# One categorical group per (subclass x tissue) combination.
cell_metadata = cell_metadata.dropna(subset=DOWNSAMPLE_GROUPBY)
emb_group = cell_metadata[DOWNSAMPLE_GROUPBY[0]].astype(str)
for col in DOWNSAMPLE_GROUPBY[1:]:
    emb_group += "+" + cell_metadata[col].astype(str)
cell_metadata["emb_group"] = emb_group.astype("category")

use_cells = []
for _, gdf in cell_metadata.groupby("emb_group", observed=True):
    if gdf.shape[0] <= MAX_CELLS_PER_GROUP:
        use_cells.extend(gdf.index)
    else:
        use_cells.extend(gdf.sample(MAX_CELLS_PER_GROUP, random_state=0).index)

train_meta = cell_metadata.loc[use_cells].copy()
train_meta["exp_batch"] = train_meta[EXP_BATCH] if EXP_BATCH else DATASET_NAME
train_meta = train_meta[
    ["sample", "cluster", "subclass", "tissue", "DissectionRegion", "exp_batch", "emb_group"]
]
train_meta.to_feather(WORK_DIR / "cell_metadata.feather")
print(f"{cell_metadata.shape[0]} cells -> {train_meta.shape[0]} sampled for training")

## Step 2 — RNA view: highly-variable genes

Load the gene-count `AnnData`, restrict it to the training cells, and keep the top
`N_TOP_GENES` highly-variable genes. Raw counts are preserved in `X` because the model uses
a count likelihood.

In [ ]:
rna_path = WORK_DIR / "adata.rna.h5ad"
if rna_path.exists():
    rna = anndata.read_h5ad(rna_path)
else:
    rna = scvi.data.read_h5ad(dataset.gene_adata_path)
    rna = rna[rna.obs_names.isin(train_meta.index)].copy()
    sc.pp.filter_genes(rna, min_counts=10)
    sc.pp.calculate_qc_metrics(rna, inplace=True, log1p=True)

    rna.layers["counts"] = rna.X.copy()          # preserve raw counts
    sc.pp.normalize_total(rna, target_sum=1e4)   # HVG selection wants normalized data...
    sc.pp.log1p(rna)
    sc.pp.highly_variable_genes(
        rna, n_top_genes=N_TOP_GENES, subset=True, layer="counts", flavor="seurat_v3"
    )
    rna.X = rna.layers.pop("counts")             # ...but the model wants raw counts back
    rna.write_h5ad(rna_path)

sc.pl.violin(rna, ["n_genes_by_counts", "total_counts"], jitter=0.4, multi_panel=True)
print(rna.shape)

## Step 3 — ATAC view: peak filter + reads → fragments

The peak matrix is stored as per-chunk `AnnData` files. Keep peaks covered in a reasonable
fraction of cells, and convert **read** counts to **fragment** counts (`ceil(reads / 2)`),
which is what the Poisson (PoissonMultiVI) likelihood expects.

> This writes a large `adata.atac.h5ad` (~1.6 GB for ChromiumPBMC) into `WORK_DIR`.

In [ ]:
atac_path = WORK_DIR / "adata.atac.h5ad"
if atac_path.exists():
    atac = anndata.read_h5ad(atac_path)
else:
    peak_paths = sorted(Path(dataset.peak_adata_path).glob("*.h5ad"))

    # Peak selection: covered above ~1% of cells (and at least the top-100k by signal).
    peak_sum, n_cells = None, 0
    for p in tqdm(peak_paths, desc="scan peaks"):
        a = anndata.read_h5ad(p)
        a = a[a.obs_names.isin(train_meta.index)]
        s = pd.Series(np.asarray(a.X.sum(0)).ravel(), index=a.var_names)
        peak_sum = s if peak_sum is None else peak_sum + s
        n_cells += a.shape[0]
    min_cov = max(n_cells * 0.01, peak_sum.sort_values(ascending=False)[:100_000].values[-1])
    use_peaks = peak_sum > min_cov
    print(f"keeping {int(use_peaks.sum())} peaks")

    parts = []
    for p in tqdm(peak_paths, desc="load peaks"):
        a = anndata.read_h5ad(p)[:, use_peaks]
        a = a[a.obs_names.isin(train_meta.index)].copy()
        parts.append(a)
    atac = anndata.concat(parts)
    atac.X.data = np.ceil(atac.X.data / 2)       # reads -> fragments
    for col, data in train_meta.items():
        atac.obs[col] = data
    atac.write_h5ad(atac_path)

print(atac.shape)

## Step 4 — Train the MultiVI embedding

Concatenate the RNA and ATAC views into one paired multiome `AnnData` and fit **MultiVI**
(`POISSONMULTIVI`). The `exp_batch` (donor) covariate is modeled so the latent space
reflects cell state rather than experimental batch.

In [ ]:
model_dir = WORK_DIR / "model"

rna = rna[atac.obs_names].copy()                 # align the two views on the same cells
adata_train = anndata.concat([rna, atac], axis=1)
for col, data in atac.obs.items():
    adata_train.obs[col] = data
adata_train.obs["modality"] = "paired"

rna_var_names = rna.var_names.copy()
atac_var_names = atac.var_names.copy()

POISSONMULTIVI.setup_anndata(
    adata_train, batch_key="modality", categorical_covariate_keys=["exp_batch"]
)

if model_dir.exists():
    model = POISSONMULTIVI.load(str(model_dir), adata=adata_train)
else:
    model = POISSONMULTIVI(
        adata_train,
        n_genes=rna_var_names.size,
        n_regions=atac_var_names.size,
        n_latent=N_LATENT,
    )
    model.train(
        accelerator="auto",
        adversarial_mixing=False,
        batch_size=512,
        max_epochs=MAX_EPOCHS,
        early_stopping=True,
    )
    model.save(str(model_dir), overwrite=True)
    joblib.dump(
        {"rna_var_names": rna_var_names, "atac_var_names": atac_var_names},
        model_dir / "features.joblib",
    )
    pd.DataFrame(
        {k: v.squeeze() for k, v in model.history.items()}
    ).to_csv(WORK_DIR / "train_history.csv")

In [ ]:
# Training curve (available whenever the model was trained in this session or cached to CSV).
hist_path = WORK_DIR / "train_history.csv"
if model.history:
    hist = pd.DataFrame({k: v.squeeze() for k, v in model.history.items()})
elif hist_path.exists():
    hist = pd.read_csv(hist_path, index_col=0)
else:
    hist = None

if hist is not None and "validation_loss" in hist:
    plt.figure(figsize=(4, 3))
    plt.plot(hist["validation_loss"])
    plt.xlabel("epoch")
    plt.ylabel("validation loss")
else:
    print("No training history (model was loaded from cache); train fresh to plot the curve.")

## Step 5 — Apply the model to *all* cells

Training used a downsampled cell list. Now compute the latent embedding for **every** cell in
the dataset, streaming over the peak chunks to keep memory bounded, and save it as a feather.

In [ ]:
emb_path = WORK_DIR / "pmvi.latent_embedding.feather"
if emb_path.exists():
    latent_emb = pd.read_feather(emb_path)
else:
    all_meta = dataset.make_cell_metadata()
    all_meta["exp_batch"] = all_meta[EXP_BATCH] if EXP_BATCH else DATASET_NAME

    rna_all = anndata.read_h5ad(dataset.gene_adata_path, backed="r")
    rna_all = rna_all[:, rna_var_names].to_memory()

    parts = []
    for p in tqdm(sorted(Path(dataset.peak_adata_path).glob("*.h5ad")), desc="infer"):
        a = anndata.read_h5ad(p)
        a = a[a.obs_names.isin(all_meta.index), atac_var_names].copy()
        a.X.data = np.ceil(a.X.data / 2)                       # reads -> fragments
        infer = anndata.concat([rna_all[a.obs_names].copy(), a], axis=1)
        infer.obs["modality"] = "paired"
        infer.obs["exp_batch"] = all_meta["exp_batch"]
        POISSONMULTIVI.setup_anndata(
            infer, batch_key="modality", categorical_covariate_keys=["exp_batch"]
        )
        z = model.get_latent_representation(adata=infer)
        parts.append(pd.DataFrame(z, index=infer.obs_names))
    latent_emb = pd.concat(parts)
    latent_emb.to_feather(emb_path)

print(latent_emb.shape)

## Step 6 — Neighbors, UMAP and a first look

Assemble an `AnnData` whose `obsm["X_pmvi"]` holds the embedding, build the kNN graph, and run
UMAP + Leiden. This `with_coords` file is the hand-off to the
[metacell page](02_meta_cell.ipynb).

In [ ]:
coords_path = WORK_DIR / "adata.pmvi.with_coords.h5ad"
if coords_path.exists():
    adata = anndata.read_h5ad(coords_path)
else:
    all_meta = dataset.make_cell_metadata()
    cells = all_meta.index.intersection(latent_emb.index)
    adata = anndata.AnnData(
        X=csr_matrix((cells.size, 0)),
        obs=all_meta.reindex(cells),
        obsm={"X_pmvi": latent_emb.reindex(cells).values},
    )
    sc.pp.neighbors(adata, use_rep="X_pmvi")
    sc.tl.umap(adata, min_dist=0.2)
    sc.tl.leiden(
        adata, key_added="pmvi_cluster", resolution=0.2, flavor="igraph", n_iterations=2
    )
    adata.write_h5ad(coords_path)

adata

In [ ]:
adata.obs["age_int"] = adata.obs["age_int"].astype(float)
sc.pl.umap(
    adata,
    color=["class", "donor", "DissectionRegion", "age_int"],
    ncols=2,
    wspace=0.3,
    size=8,
)

---

The embedding is now saved (`adata.pmvi.with_coords.h5ad` and
`pmvi.latent_embedding.feather`). Continue to **[Meta cells](02_meta_cell.ipynb)** to collapse
cells into SEACells metacells — the units Bolero actually trains on.